In [1]:
import tensorflow as tf
print(tf.__version__)

2.13.0


# `Note`:
The goal of this session is not to attain the highest accuracy, but to get an basic understanding how `RNN` works.

The data in `RNN` will be text and output as 0 and 1 for sentiment analysis where 0 means Negative and 1 means Positive

The first step is to convert this text data into numbers or vectors bcz neural networks does not understand text data.

Here, we will learn two popular techniques to convert text data to numbers

1. Integer encoding and
2. Enbeddings


1. `Integer Encoding:

Supose we have two sentences, `Hi there` and `How are ypu`. <br>
The first step here is, we form a vocabulary of uniques words. Here, we have 5 unique words. After that we provide integer value to these words. <br>
`Hi` = 1 <br>
`there` = 2 <br>
`How` = 3 <br>
`are` = 4 <br>
`you` = 5 <br>

Thern these sentences are replaced by their integers. <br>
`Hi there` = [1, 2] <br>
`How are ypu` = [3, 4, 5]

We can observe that in first sentence. there are 2 inputs and in second sentence there are 3 inputs. So what we do to make the inputs same is, we apply `Zero padding`. Means, we add 0 at the end or at the beginning of first sentence. <br>
`Hi there` = [1, 2, 0] or [0, 1, 2] <br>
`How are ypu` = [3, 4, 5]

`Now, let's see how all these things are performed in keras.`

In [2]:
import numpy as np

In [3]:
sen = ['India wins',
      'India is the best',
      'India won',
      'India is number one',
      'Captain Dhoni',
      'Dhoni MS',
      'Kohli Virat',
      'Sachin rocks',
      'Bharat mata ki jai']

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [5]:
tokenize = Tokenizer(oov_token = '<nothing>')

# oov_token = '<nothing>' means if any user enters such word which is not present in our dict
# then then word will be treated as <nothing>

In [6]:
tokenize.fit_on_texts(sen)

# fitting the sentences

In [7]:
tokenize.word_index    # Returns index of words

{'<nothing>': 1,
 'india': 2,
 'is': 3,
 'dhoni': 4,
 'wins': 5,
 'the': 6,
 'best': 7,
 'won': 8,
 'number': 9,
 'one': 10,
 'captain': 11,
 'ms': 12,
 'kohli': 13,
 'virat': 14,
 'sachin': 15,
 'rocks': 16,
 'bharat': 17,
 'mata': 18,
 'ki': 19,
 'jai': 20}

In [8]:
tokenize.word_counts    # Returns frequency of words

OrderedDict([('india', 4),
             ('wins', 1),
             ('is', 2),
             ('the', 1),
             ('best', 1),
             ('won', 1),
             ('number', 1),
             ('one', 1),
             ('captain', 1),
             ('dhoni', 2),
             ('ms', 1),
             ('kohli', 1),
             ('virat', 1),
             ('sachin', 1),
             ('rocks', 1),
             ('bharat', 1),
             ('mata', 1),
             ('ki', 1),
             ('jai', 1)])

In [9]:
tokenize.document_count    # Returns sentences in the list. 

9

In [10]:
len(sen)

9

In [11]:
sequence = tokenize.texts_to_sequences(sen)

sequence

# This converts words with their indices

# [2, 5] means [India wins]
# [2, 3, 6, 7] means [India is the best]


[[2, 5],
 [2, 3, 6, 7],
 [2, 8],
 [2, 3, 9, 10],
 [11, 4],
 [4, 12],
 [13, 14],
 [15, 16],
 [17, 18, 19, 20]]

In [12]:
# We can see that the len of sequences are different, so, we will apply padding.
# Padding means we will insert 0 at the end or in the beginning

from tensorflow.keras.utils import pad_sequences

In [13]:
sequence = pad_sequences(sequence, padding = 'pre')

# post means at the end
# pre means in the beginning

sequence

array([[ 0,  0,  2,  5],
       [ 2,  3,  6,  7],
       [ 0,  0,  2,  8],
       [ 2,  3,  9, 10],
       [ 0,  0, 11,  4],
       [ 0,  0,  4, 12],
       [ 0,  0, 13, 14],
       [ 0,  0, 15, 16],
       [17, 18, 19, 20]])

# This is how we preprocess our input data in RNN.
# Now, we will built the whole model to get a view how things work.

In [14]:
# from tensorflow import keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten, SimpleRNN, Embedding

In [15]:
(x_train, y_train), (x_test, y_test) = imdb.load_data()

In [16]:
x_train[0]

# Here, we are getting indices of the words rather than the sentences bcz the data is aready tokenized and integer encoded

[1,
 14,
 22,
 16,
 43,
 530,
 973,
 1622,
 1385,
 65,
 458,
 4468,
 66,
 3941,
 4,
 173,
 36,
 256,
 5,
 25,
 100,
 43,
 838,
 112,
 50,
 670,
 22665,
 9,
 35,
 480,
 284,
 5,
 150,
 4,
 172,
 112,
 167,
 21631,
 336,
 385,
 39,
 4,
 172,
 4536,
 1111,
 17,
 546,
 38,
 13,
 447,
 4,
 192,
 50,
 16,
 6,
 147,
 2025,
 19,
 14,
 22,
 4,
 1920,
 4613,
 469,
 4,
 22,
 71,
 87,
 12,
 16,
 43,
 530,
 38,
 76,
 15,
 13,
 1247,
 4,
 22,
 17,
 515,
 17,
 12,
 16,
 626,
 18,
 19193,
 5,
 62,
 386,
 12,
 8,
 316,
 8,
 106,
 5,
 4,
 2223,
 5244,
 16,
 480,
 66,
 3785,
 33,
 4,
 130,
 12,
 16,
 38,
 619,
 5,
 25,
 124,
 51,
 36,
 135,
 48,
 25,
 1415,
 33,
 6,
 22,
 12,
 215,
 28,
 77,
 52,
 5,
 14,
 407,
 16,
 82,
 10311,
 8,
 4,
 107,
 117,
 5952,
 15,
 256,
 4,
 31050,
 7,
 3766,
 5,
 723,
 36,
 71,
 43,
 530,
 476,
 26,
 400,
 317,
 46,
 7,
 4,
 12118,
 1029,
 13,
 104,
 88,
 4,
 381,
 15,
 297,
 98,
 32,
 2071,
 56,
 26,
 141,
 6,
 194,
 7486,
 18,
 4,
 226,
 22,
 21,
 134,
 476,
 26,
 480,
 5

In [17]:
# Here, data is tokenized but padding is not done. Each sentence is of different size.
# So we will have to apply padding and make them of same siz



In [18]:
print(len(x_train[0]))
print(len(x_train[1]))
print(len(x_train[2]))

218
189
141


In [19]:
data = [x_train, x_test]
padded_data = [pad_sequences(x, padding='post', maxlen = 50) for x in data]

x_train, x_test= padded_data

# maxlen = 50 means give me 50 words from each review.
# I am doing this bcz if not, will take much time to train the model

In [20]:
print(len(x_train[0]))
print(len(x_train[1]))
print(len(x_train[2]))

50
50
50


In [21]:
x_train.shape

# (25000, 50) means it has 25000 sentences i.e reviews and each of 50 words

(25000, 50)

In [22]:
x_test.shape

(25000, 50)

# Building model

In [23]:
model = Sequential()

model.add(SimpleRNN(32, input_shape = (50, 1), return_sequences = False))
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn (SimpleRNN)      (None, 32)                1088      
                                                                 
 dense (Dense)               (None, 1)                 33        
                                                                 
Total params: 1121 (4.38 KB)
Trainable params: 1121 (4.38 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [24]:
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [25]:
history = model.fit(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Epoch 1/5
782/782 [==============================] - 7s 7ms/step - loss: 0.6943 - accuracy: 0.5092 - val_loss: 0.6957 - val_accuracy: 0.5052
Epoch 2/5
782/782 [==============================] - 5s 7ms/step - loss: 0.6932 - accuracy: 0.5027 - val_loss: 0.6937 - val_accuracy: 0.5073
Epoch 3/5
782/782 [==============================] - 6s 7ms/step - loss: 0.6928 - accuracy: 0.5085 - val_loss: 0.6953 - val_accuracy: 0.5029
Epoch 4/5
782/782 [==============================] - 6s 7ms/step - loss: 0.6928 - accuracy: 0.5068 - val_loss: 0.6947 - val_accuracy: 0.5063
Epoch 5/5
782/782 [==============================] - 5s 6ms/step - loss: 0.6927 - accuracy: 0.5035 - val_loss: 0.6947 - val_accuracy: 0.5080


# Using Embedding layer

In [26]:
(x_train2, y_train2), (x_test2, y_test2) = imdb.load_data(num_words = 10000)

# num_words = 10000 means take top 10000 words with hightest frequency
# Means it creates a dictionary of top 10000 words with highest frequency

In [27]:
data = [x_train2, x_test2]
padded_data = [pad_sequences(x, padding='post', maxlen = 50) for x in data]

x_train2, x_test2= padded_data

# maxlen = 50 means give me 50 words from each review.
# I am doing this bcz if not, will take much time to train the model

In [28]:
x_train2.shape

(25000, 50)

In [29]:
model2 = Sequential()
model2.add(Embedding(10000, output_dim = 2,input_length = 50))
model2.add(SimpleRNN(32,return_sequences=False))
model2.add(Dense(1, activation='sigmoid'))

model2.summary()

# Embedding(10000, output_dim = 2,input_length = 50)

# 10000 means we are having a dict of 10000 unique words
# output_dim = 2 means it coverts index given to words into a vector of 2 numbers
# For example: Index of good is 5, then it gets converted to [0.5, 0.8]. Now this vector indicates 'good', not index 5
# Index of 'bad' is 7, then it may get converted to [0.7, -0.5].

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 50, 2)             20000     
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, 32)                1120      
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 21153 (82.63 KB)
Trainable params: 21153 (82.63 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [30]:
model2.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [31]:
history2 = model2.fit(x_train2, y_train2, epochs = 5, validation_data = (x_test2, y_test2))

# See, acuracy increased just by using enbedding layer

Epoch 1/5
782/782 [==============================] - 9s 10ms/step - loss: 0.5891 - accuracy: 0.6571 - val_loss: 0.4530 - val_accuracy: 0.7904
Epoch 2/5
782/782 [==============================] - 7s 9ms/step - loss: 0.3824 - accuracy: 0.8354 - val_loss: 0.4316 - val_accuracy: 0.8020
Epoch 3/5
782/782 [==============================] - 7s 9ms/step - loss: 0.3106 - accuracy: 0.8731 - val_loss: 0.4320 - val_accuracy: 0.8048
Epoch 4/5
782/782 [==============================] - 7s 9ms/step - loss: 0.2638 - accuracy: 0.8964 - val_loss: 0.4523 - val_accuracy: 0.8024
Epoch 5/5
782/782 [==============================] - 7s 8ms/step - loss: 0.2246 - accuracy: 0.9158 - val_loss: 0.5557 - val_accuracy: 0.7940


In [32]:
x_train.shape

(25000, 50)

In [33]:
y_train.shape

(25000,)

In [34]:
x_test.shape

(25000, 50)

# Let's see if enbedding layer returns numbers as asked for

In [35]:
sen = ['Nikunj Bosamiya',
       'Jugal Bosamiya',
       'Harshit Kamani',
       'Jenish Mer',
       'Parth Paghdar'
      ]

In [36]:
tokenize2 = Tokenizer(oov_token = '<unknown>')

In [37]:
tokenize2.fit_on_texts(sen)


In [38]:
tokenize2.word_index

{'<unknown>': 1,
 'bosamiya': 2,
 'nikunj': 3,
 'jugal': 4,
 'harshit': 5,
 'kamani': 6,
 'jenish': 7,
 'mer': 8,
 'parth': 9,
 'paghdar': 10}

In [39]:
tokenize2.word_counts

OrderedDict([('nikunj', 1),
             ('bosamiya', 2),
             ('jugal', 1),
             ('harshit', 1),
             ('kamani', 1),
             ('jenish', 1),
             ('mer', 1),
             ('parth', 1),
             ('paghdar', 1)])

In [40]:
# Converting to Integer encoding (Index encoding)

sequence =  tokenize2.texts_to_sequences(sen)

sequence

[[3, 2], [4, 2], [5, 6], [7, 8], [9, 10]]

In [41]:
# padding
max_len = max(len(x) for x in sequence)
print(max_len)

padded_seq = pad_sequences(sequence, maxlen = max_len, padding = 'post')

print()
print()

print(padded_seq)    # No 0 are adden at last bcz each item is of 2 words

2


[[ 3  2]
 [ 4  2]
 [ 5  6]
 [ 7  8]
 [ 9 10]]


In [42]:
# Defining the embedding layer

vocab_len = len(tokenize2.word_index) + 1
emb_dim = 2    # We want 2 numbers for each word. So in total, we will have 20 numbers

emb_layer = Embedding(input_dim = vocab_len, output_dim = 2)

In [43]:
emb_output = emb_layer(padded_seq)

In [44]:
emb_output.numpy()

array([[[ 0.04102032,  0.04998335],
        [-0.04094703,  0.0004608 ]],

       [[ 0.03214066, -0.02115209],
        [-0.04094703,  0.0004608 ]],

       [[ 0.03275427, -0.01860296],
        [ 0.02358892,  0.02613641]],

       [[-0.04283887,  0.00053284],
        [ 0.04285053, -0.02494564]],

       [[ 0.02372565, -0.00686435],
        [-0.00417047,  0.03537093]]], dtype=float32)

# Let;s take opposite words

In [45]:
sen = ['Good Bad',
       'Heavy Light',
       'Day Night',
       'Easy Tough',
       'Alive Death'
      ]

In [46]:
tokenize2 = Tokenizer(oov_token = '<unknown>')

In [47]:
tokenize2.fit_on_texts(sen)

In [48]:
tokenize2.word_index

{'<unknown>': 1,
 'good': 2,
 'bad': 3,
 'heavy': 4,
 'light': 5,
 'day': 6,
 'night': 7,
 'easy': 8,
 'tough': 9,
 'alive': 10,
 'death': 11}

In [49]:
tokenize2.word_counts

OrderedDict([('good', 1),
             ('bad', 1),
             ('heavy', 1),
             ('light', 1),
             ('day', 1),
             ('night', 1),
             ('easy', 1),
             ('tough', 1),
             ('alive', 1),
             ('death', 1)])

In [50]:
# Converting to Integer encoding (Index encoding)

sequence =  tokenize2.texts_to_sequences(sen)

sequence

[[2, 3], [4, 5], [6, 7], [8, 9], [10, 11]]

In [51]:
# padding
max_len = max(len(x) for x in sequence)
print(max_len)

padded_seq = pad_sequences(sequence, maxlen = max_len, padding = 'post')

print()
print()

print(padded_seq)    # No 0 are adden at last bcz each item is of 2 words

2


[[ 2  3]
 [ 4  5]
 [ 6  7]
 [ 8  9]
 [10 11]]


In [52]:
# Defining the embedding layer

vocab_len = len(tokenize2.word_index) + 1
emb_dim = 2    # We want 2 numbers for each word. So in total, we will have 20 numbers

emb_layer = Embedding(input_dim = vocab_len, output_dim = 2)

In [53]:
emb_output = emb_layer(padded_seq)

In [54]:
emb_output.numpy()

array([[[ 0.01494913, -0.00987859],
        [ 0.02872879,  0.00074004]],

       [[ 0.0261889 ,  0.0328483 ],
        [ 0.03694035, -0.01043593]],

       [[-0.01139066, -0.02799972],
        [ 0.01586613,  0.00836081]],

       [[ 0.03967757, -0.00899936],
        [ 0.02979464, -0.03630716]],

       [[ 0.00820227,  0.01798851],
        [ 0.03924953,  0.04032507]]], dtype=float32)

In [55]:
# See, we are getting exact opposite numbers for opposite words

# End of the session.